In [5]:
import pandas as pd
import numpy as np

def process_single_dataset(df, text_col, category_col, samples_per_category=40, min_review_length=50, random_state=42):
    """
    Processes a single dataset to create a balanced sample.
    Ensures exactly `samples_per_category` reviews per category (or max available), prioritizing longer reviews.
    
    Args:
        df (pd.DataFrame): The input dataset.
        text_col (str): The name of the column containing the review text.
        category_col (str): The name of the column containing the category labels.
        samples_per_category (int): Number of samples to draw per category.
        min_review_length (int): Minimum word count to prioritize a review.
        random_state (int): Seed for reproducible random sampling.
    
    Returns:
        pd.DataFrame: A new DataFrame containing the balanced sample for this dataset.
    """
    np.random.seed(random_state)
    final_samples = []

    # Step 1: Handle missing values
    df = df.dropna(subset=[text_col]).copy()
    df['review_length'] = df[text_col].apply(lambda x: len(str(x).split()))

    # Step 2: Process each category
    categories = df[category_col].unique()
    
    for cat in categories:
        subset = df[df[category_col] == cat]
        
        # First, try to get N long reviews
        long_reviews = subset[subset['review_length'] >= min_review_length]
        n_long = len(long_reviews)
        
        if n_long >= samples_per_category:
            sampled = long_reviews.sample(n=samples_per_category, random_state=random_state)
        else:
            # If not enough long reviews, take all long + remaining from shorter
            remaining_needed = samples_per_category - n_long
            short_reviews = subset[subset['review_length'] < min_review_length]
            
            if len(short_reviews) >= remaining_needed:
                short_sampled = short_reviews.sample(n=remaining_needed, random_state=random_state)
            else:
                short_sampled = short_reviews 
            
            sampled = pd.concat([long_reviews, short_sampled])
        
        final_samples.append(sampled)

    # Combine all samples
    final_dataset = pd.concat(final_samples).reset_index(drop=True)
    
    # Print summary for this dataset
    print(f"Final sample size for this dataset: {len(final_dataset)}")
    print("Distribution across categories:\n", final_dataset[category_col].value_counts())
    print("-" * 50)
    
    return final_dataset




# Load three datasets
df1 = pd.read_csv('auto_samplingApple_store.csv')  
df2 = pd.read_csv('auto_samping_amazon.csv') 
df3 = pd.read_csv('auto_samplingGP_store.csv')

# Process each dataset independently
final_sample_1 = process_single_dataset(
    df1,
    text_col="text_review",
    category_col="apps_category",
    samples_per_category=40,
    min_review_length=50,
    random_state=42
)

final_sample_2 = process_single_dataset(
    df2,
    text_col="text_review",
    category_col="apps_category",
    samples_per_category=40,
    min_review_length=50,
    random_state=42
)

final_sample_3 = process_single_dataset(
    df3,
    text_col="text_review",
    category_col="apps_category",
    samples_per_category=40,
    min_review_length=50,
    random_state=42
)

# Save the three final sample datasets
final_sample_1.to_csv("final_sample_appledataset.csv", index=False)
final_sample_2.to_csv("final_sample_amazondataset.csv", index=False)
final_sample_3.to_csv("final_sample_gplaydataset.csv", index=False)

print("All three datasets have been processed and saved successfully.")

Final sample size for this dataset: 480
Distribution across categories:
 apps_category
Social Networking    40
Finance              40
Entertainment        40
Refrences            40
Social networking    40
Business             40
Education            40
Health & Fitness     40
lifeStyle            40
Sports               40
Weather              40
Travel               40
Name: count, dtype: int64
--------------------------------------------------
Final sample size for this dataset: 560
Distribution across categories:
 apps_category
communication      40
education          40
food_and_drinks    40
game               40
Lifestyle          40
movie_and_tv       40
music_and_audio    40
novelty            40
photo_and_video    40
Travel             40
business           40
productivity       40
sports             40
utilities          40
Name: count, dtype: int64
--------------------------------------------------
Final sample size for this dataset: 640
Distribution across categories:
 app